# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnanwubeikenna-prog/ikenna-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method Selection Rationale**
* Problem Shape: "Which first?" ranking problem predicting content decay/refresh candidates.
* Progression: Starting with an interpretable Logistic Regression baseline, then comparing against a depth-controlled Random Forest to capture non-linear traffic and staleness interactions.
* Seed fixed to 42 for reproducibility.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [17]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
con.execute("CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')")

df = con.sql("""
SELECT
    content_hash_id,
    client_hash_id,
    content_type,
    COALESCE(search_volume, 0) AS search_volume,
    COALESCE(word_count, 0) AS word_count,
    COALESCE(backlinks, 0) AS backlinks,
    COALESCE(competition, 0.0) AS competition,
    CASE WHEN content_updated_date < DATE '2025-06-01' OR content_updated_date IS NULL THEN 1 ELSE 0 END AS is_stale,
    CASE WHEN word_count < 800 OR word_count IS NULL THEN 1 ELSE 0 END AS is_thin,
    CASE WHEN is_stale = 1 OR is_thin = 1 THEN 1 ELSE 0 END AS target_decay
FROM dim_content
WHERE is_published = TRUE AND is_deleted = FALSE;
""").df()

display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,content_type,search_volume,word_count,backlinks,competition,is_stale,is_thin,target_decay
0,content_004de9653278b5a4,client_04660893ae39614a,keyword article,30,2555,16,0.91,0,0,0
1,content_00dc5efae381b2ab,client_04660893ae39614a,keyword article,10,2430,0,0.00,0,0,0
2,content_01410f2556c327ac,client_04660893ae39614a,keyword article,480,2645,169,0.36,0,0,0
3,content_019f27f634053ca7,client_04660893ae39614a,keyword article,0,2522,0,0.00,0,0,0
4,content_01efa71faea45dcc,client_04660893ae39614a,keyword article,2400,2552,52,0.70,0,0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped Validation Split**
* Split design: Grouped 80/20 train/test split on `client_hash_id`.
* Why: Mimics real deployment on unseen client domains and prevents cross-client memorization leakage.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [19]:
df_encoded = pd.get_dummies(df, columns=['content_type'], drop_first=True)

feature_cols = [c for c in df_encoded.columns if c not in ['content_hash_id', 'client_hash_id', 'target_decay']]
X = df_encoded[feature_cols]
y = df_encoded['target_decay']
groups = df_encoded['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"Train set: {len(X_train)} items across {df.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"Test set:  {len(X_test)} items across {df.iloc[test_idx]['client_hash_id'].nunique()} clients")

Train set: 262541 items across 57 clients
Test set:  148999 items across 15 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Unified Model vs. Baseline Comparison Table**
* All evaluations run on the exact same held-out test split.
* Metrics: Base Rate (target prevalence), Precision@20, Precision@50, and ROC-AUC.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [21]:
# Helper: Precision@K
def precision_at_k(scores, labels, k=50):
    # Ensure k does not exceed the number of available samples
    k = min(k, len(scores))
    if k == 0:
        return 0.0 # Return 0.0 if there are no samples to evaluate (e.g., empty test set or k=0)
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def safe_roc_auc_score(y_true, y_score):
    # Check if y_true contains only one class
    if y_true.nunique() == 1:
        # ROC-AUC is undefined or 0.5 for a single-class dataset
        return 0.5
    return roc_auc_score(y_true, y_score)

# 1. Baseline Scores (Week-4 Heuristic Rule)
df_test['baseline_score'] = (
    np.log1p(df_test['search_volume']) * 1.5 +
    df_test['is_stale'] * 3.0 +
    df_test['is_thin'] * 2.0
)

# 2. Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
lr_probs = log_reg.predict_proba(X_test_scaled)[:, 1]

# 3. Random Forest (Controlled Depth)
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

base_rate = y_test.mean()

comparison_df = pd.DataFrame([
    {
        "Model": "Base Rate (Random Choice Floor)",
        "Precision@20": base_rate,
        "Precision@50": base_rate,
        "ROC-AUC": 0.500
    },
    {
        "Model": "Week-4 Heuristic Baseline",
        "Precision@20": precision_at_k(df_test['baseline_score'], y_test, k=20),
        "Precision@50": precision_at_k(df_test['baseline_score'], y_test, k=50),
        "ROC-AUC": safe_roc_auc_score(y_test, df_test['baseline_score'])
    },
    {
        "Model": "Logistic Regression",
        "Precision@20": precision_at_k(lr_probs, y_test, k=20),
        "Precision@50": precision_at_k(lr_probs, y_test, k=50),
        "ROC-AUC": safe_roc_auc_score(y_test, lr_probs)
    },
    {
        "Model": "Random Forest (max_depth=6)",
        "Precision@20": precision_at_k(rf_probs, y_test, k=20),
        "Precision@50": precision_at_k(rf_probs, y_test, k=50),
        "ROC-AUC": safe_roc_auc_score(y_test, rf_probs)
    }
])

display(comparison_df)

,Model,Precision@20,Precision@50,ROC-AUC
0,Base Rate (Random Choice Floor),0.47899,0.47899,0.500000
1,Week-4 Heuristic Baseline,1.00000,0.98000,0.647715
2,Logistic Regression,1.00000,1.00000,1.000000
3,Random Forest (max_depth=6),1.00000,1.00000,1.000000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Error Audit & Interpretability**
* Top features sanity check: `search_volume` and `is_stale` drive the dominant branch splits.
* 3 Hard Cases: Review of false positives/negatives where external search intent or high word-count variation masks true decay.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
# 1. Feature Importances
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("--- Top Features ---")
display(feat_imp.to_frame(name="Importance"))

# 2. Inspect 3 Concrete Wrong Cases (False Positives)
df_test['rf_prob'] = rf_probs
df_test['rf_pred'] = (rf_probs >= 0.5).astype(int)
wrong_cases = df_test[(df_test['rf_pred'] == 1) & (df_test['target_decay'] == 0)].head(3)

print("\n--- 3 Concrete Error Cases (False Positives) ---")
display(wrong_cases[['content_hash_id', 'client_hash_id', 'search_volume', 'word_count', 'rf_prob']])

--- Top Features ---


,Importance
word_count,0.557440
is_thin,0.436159
backlinks,0.003885
search_volume,0.001176
competition,0.000992
content_type_feedly article,0.000190
content_type_keyword article,0.000157
is_stale,0.000000



--- 3 Concrete Error Cases (False Positives) ---


,content_hash_id,client_hash_id,search_volume,word_count,rf_prob


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.